# CUB B-0에서 Taylor pruning과 CE+KL 회복 학습

CUB-200-2011로 학습한 Git B-0에서 30%, 50%, 70% Teacher를 순서대로 만든다. 비율은 원래 ConvNeXt MLP hidden channel의 누적 제거율이며 전체 parameter 제거율과 다르다. target별 best와 last를 각각 보존한다.

train batch16, effective batch64, B-0 및 모든 Teacher Validation/Test batch8을 고정한다. B-0는 epoch10, 200-class head이며 raw SHA는 `3ceffe27b7d41a711937961e117a28d95b2d5611be4dc473969155b78620f207`이다. B-0를 재학습하지 않는다.

GitHub에서 노트북만 Colab으로 열면 저장소 파일은 복사되지 않는다. 먼저 공유 브랜치를 `/content/ondeviceAI`에 clone하거나 저장소를 Colab 로컬 폴더에 준비하고 `REPO_ROOT`를 그 위치로 설정한다. Teacher와 KD는 서로 다른 전용 Python 커널에서 실행한다.

1. 첫 코드 셀에서 저장소, Drive 프로젝트 루트와 입력 경로를 설정한다.
2. setup 셀은 저장소의 동결 ZIP을 우선 사용하고 SHA-256, code ID와 source manifest를 검증한다. 저장소에 ZIP이 없으면 정확한 기존 Drive ZIP을 사용한다.
3. 계획을 확인한 뒤 필요한 실행 옵션을 명시적으로 켠다. 공유 기본값은 `START_TRAINING=False`, `AUTO_RELEASE=False`이다.

원본 이미지와 가중치는 ZIP에 포함하지 않는다. 원래 Drive source와 기존 Native baseline을 읽는 계약이 있으므로 Drive 프로젝트 루트만 바꾸는 것으로 다른 계정의 입력이 준비되지는 않는다. source ID와 해시가 다르면 실행을 중단한다.

## 1. 경로와 실행 옵션

`DRIVE_PROJECT_ROOT` 아래 `TeacherSizeKD_NoPruning/inputs`의 입력을 사용한다. `DATA_ROOT=None`은 패키지의 검증된 준비 경로를 사용하며 기존 CUB 이미지 폴더가 있으면 직접 지정할 수 있다. 이미지, split, 클래스 순서 검증을 통과해야 한다.

새 실행은 `RESUME=False`, 동일 계약의 중단 재개는 `RESUME=True`를 사용한다. `START_TRAINING=True`는 학습 진입, `EVALUATE_SAVED_RESULTS=True`는 별도 Test 진입이다. 필요한 실행 셀만 사용한다.

In [ ]:
from pathlib import Path
REPO_ROOT = Path('/content/ondeviceAI')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/온디바이스AI')
START_TRAINING = False
RESUME = False
RUN_TEST_AFTER_TRAINING = True
AUTO_RELEASE = False
RUN_NAME = 'GitB0_TaylorCEKL_30_50_70_v1_2'
EVALUATION_BATCH_SIZE = 8
assert EVALUATION_BATCH_SIZE == 8
EXPECTED_B0_SHA256 = '3ceffe27b7d41a711937961e117a28d95b2d5611be4dc473969155b78620f207'
DATA_ROOT = None
PAUSE_AFTER_TARGET = None
STOP_AFTER_SEGMENT_EPOCH = None
RUNTIME_RELEASE_EVIDENCE = None
EVALUATE_SAVED_RESULTS = False
OUTPUT_FOLDER_ID = None
assert PAUSE_AFTER_TARGET in (None,30,50,70)
assert STOP_AFTER_SEGMENT_EPOCH is None or 1<=STOP_AFTER_SEGMENT_EPOCH<=40
print('OPTIONS_VERIFIED; Teacher80epochs; START',START_TRAINING,'Test_after_all',RUN_TEST_AFTER_TRAINING)

## 2. 동결 runtime과 계획 검증

Drive를 연결하고 정확한 ZIP을 Colab 로컬 임시 폴더에 푼다. source hash, code ID, 전용 커널과 Native bootstrap을 검증한 뒤 계획을 출력한다. 이 셀에서는 학습이나 Test 함수를 호출하지 않는다.

In [ ]:
from pathlib import Path
from google.colab import drive, auth
import os, sys

REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
BASE = Path(DRIVE_PROJECT_ROOT).expanduser()
if not (REPO_ROOT / 'experiments/cub_taylor_kd/bootstrap.py').is_file():
    raise FileNotFoundError('공유 브랜치를 Colab에 clone하고 REPO_ROOT를 확인하세요.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from experiments.cub_taylor_kd.bootstrap import load_runtime
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive', timeout_ms=900000)
B0_CODE = load_runtime('teacher', repo_root=REPO_ROOT, drive_project_root=BASE)
from lightio.contracts import verify_delivery
assert verify_delivery(B0_CODE)['code_id'] == '81aefa8e76aa6e245c427745c532d0545ebd072d14669da21e0a057cdcde5a7d'
from lightio.runtime_bootstrap import cub_runtime_bindings
cfg, _, _ = cub_runtime_bindings()
from lightio.eval_policy import EVALUATION_BATCH_SIZE as PINNED_EVAL_BATCH_SIZE
assert EVALUATION_BATCH_SIZE == PINNED_EVAL_BATCH_SIZE == 8
assert cfg['batch_size'] == 16 and cfg['effective_batch_size'] == 64
from b1kd.runtime import current_training_runtime
from lightio.extend_runner import plan, run
assert Path(run.__code__.co_filename).resolve().is_relative_to(B0_CODE.resolve())
print('GIT_B0_CODE_VERIFIED', '81aefa8e76aa6e245c427745c532d0545ebd072d14669da21e0a057cdcde5a7d', str(B0_CODE))
print(plan())

## 3. Teacher pruning과 회복 학습

train-only calibration은 클래스별2장, 총400장이다. 현재 가중치에서 Taylor 중요도를 계산하고 각 pruning boundary에서 새 AdamW와 scaler를 생성한다. 손실은 `CE + 0.5 × T² × KL`, `T=4`이며 고정 원본 B-0를 FP32 KL Teacher로 사용한다. seed20261013, weight decay0.05, backbone/head LR1e-5/1e-4, 단계별 warmup2와 cosine을 유지한다.

7.5/15/22.5/30%는 각각10epoch, 35/40/45/50/55/60/65/70%는 각각5epoch이며 총80 Teacher epochs이다. 다음 target은 이전 LAST에서 출발한다. target별 마지막 quarter의 Validation OA, CE, 이른 epoch 순서로 best를 고른다.

B-0 Validation에서 Hard40을 고정한다. `RUN_TEST_AFTER_TRAINING=True`이면 학습 완료와 선정본 잠금 후 B0/30/50/70 Test를 평가한다. Test로 재선정하지 않으며 이미 관찰한 CUB Test는 새로운 holdout이 아니다. report의 folder ID, checkpoint receipt와 selected validation을 보존한다. `AUTO_RELEASE=True`를 선택한 경우에만 최종 저장 후 runtime 반환을 요청한다.

In [ ]:
if START_TRAINING:
    auth.authenticate_user()
    from types import SimpleNamespace
    GIT_B0_REPORT=run(SimpleNamespace(start_training=True,resume=RESUME,auto_release=AUTO_RELEASE,
        test_after_training=RUN_TEST_AFTER_TRAINING,expected_B0_sha=EXPECTED_B0_SHA256,
        pause_after_target=PAUSE_AFTER_TARGET,stop_after_epoch=STOP_AFTER_SEGMENT_EPOCH,
        run_name=RUN_NAME,data_root=DATA_ROOT,runtime_release_evidence=RUNTIME_RELEASE_EVIDENCE,
        inputs=str(BASE/'TeacherSizeKD_NoPruning/inputs')))
    print(GIT_B0_REPORT)
else:
    print('NO_TRAINING_OR_TEST_STARTED: newGPU0/newepoch0')

## 4. 결과 읽기와 별도 Test 재개

학습 셀에서 Test까지 마쳤으면 추가 실행하지 않는다. 학습만 마쳤거나 Test가 중단된 경우 `EVALUATE_SAVED_RESULTS=True`, `RESUME=True`와 반환된 `OUTPUT_FOLDER_ID`를 설정하고 setup과 아래 셀을 실행한다.

각 target 폴더의 `best.pth`, `config.json`, `github_history.json`, `pruning_spec.json`과 completion/selection receipt를 함께 읽는다. 모델은 동결 패키지의 `restore_export` adapter로 복원한다. best와 LAST의 용도는 다르며 다음 target의 출발점은 LAST이다.

progress/history는 매epoch, checkpoint는 local atomic 및 Drive900초와 stage/final/pause flush를 사용한다. computed와 durable을 구분하고15분 고정 손실 상한을 가정하지 않는다. owner/lock을 삭제하지 않는다. 재개할 때 pause/stop을 해제하고 필요한 runtime release evidence를 제공한다.

In [ ]:
if EVALUATE_SAVED_RESULTS:
    auth.authenticate_user()
    import json,google.auth
    from googleapiclient.discovery import build
    from lightio.identity import identity,read
    from lightio.backend import DriveBackend
    from lightio.extend_test import evaluate
    credentials,_=google.auth.default()
    service=build('drive','v3',credentials=credentials,cache_discovery=False)
    saved=globals().get('GIT_B0_REPORT')
    if saved is None:
        assert OUTPUT_FOLDER_ID,'Set the printed output folder ID; never repeat training'
        rows=service.files().list(q=f"'{OUTPUT_FOLDER_ID}' in parents and name='OPERATIONAL_CONTRACT.json' and trashed=false",fields='files(id),nextPageToken').execute()
        assert len(rows.get('files',[]))==1 and not rows.get('nextPageToken')
        op=json.loads(service.files().get_media(fileId=rows['files'][0]['id']).execute())
        assert identity({k:v for k,v in op.items() if k!='operational_id'})==op['operational_id']
        reader=DriveBackend(OUTPUT_FOLDER_ID,op['operational_id'])
        assert json.loads(reader.get('OWNER.json'))['state']=='complete'
        saved=json.loads(reader.get('COMPLETE.json'))
    released=read(RUNTIME_RELEASE_EVIDENCE) if RUNTIME_RELEASE_EVIDENCE else {}
    GIT_B0_TEST_REPORT=evaluate(saved,B0_CODE,service,inputs=str(BASE/'TeacherSizeKD_NoPruning/inputs'),
        data_root=DATA_ROOT,resume=RESUME,request_release=AUTO_RELEASE,prior_release=released.get('Test'))
    print(GIT_B0_TEST_REPORT)
    if AUTO_RELEASE:
        from google.colab import runtime
        runtime.unassign()
else:
    print('SEPARATE_TEST_NOT_REQUESTED')